# BC카드 시장 침투도 기반 공략 우선순위 발굴

**제1회 AI금융빅데이터플랫폼 소비데이터 공모전** · 팀 유비온
팀장 이성진 · 팀원 염신호 · 이규상

전국 227개 시군구 × 8개 업종의 **1,813개 시장**에서 "이 정도 점포 수와 인구라면 나와야 할
BC 결제액"을 회귀로 추정하고, 실제와의 차이로 공략 우선순위를 매긴다.

| 절 | 내용 | 결과 |
|---|---|---|
| 1 | 데이터 결합 | 1,813개 조합 |
| 2 | 업종별 회귀 → 침투지수 | 가중평균 R² 0.891 |
| 3 | σ — 모델 오차 대비 이탈 정도 | 업종별 예측오차 |
| 4 | 4단 필터 → 후보 선정 | **12개 · 1,059억** |
| 5 | 왜 약한가 — 건수 · 건당 분해 | 건수가 문제 |
| 6 | 기회 유형 3가지 | 7 / 4 / 1 |
| 7 | 검증 — 홀드아웃 3종 · 컷오프 민감도 | 12/12 유지 |

> 이 노트북은 분석 본류만 담았다. 그림 생성과 탐색 과정은 별도 노트북에 있다.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

pd.set_option("display.max_columns", 50)

def 표(df, 제목="", 포맷=None):
    """DataFrame을 제목 붙은 표로 보기 좋게 출력한다."""
    s = (df.style.hide(axis="index").set_caption(제목)
           .set_table_styles([
               {"selector": "caption", "props": [("caption-side", "top"), ("text-align", "left"),
                    ("font-size", "13.5px"), ("font-weight", "700"),
                    ("padding", "8px 0 6px"), ("color", "#1f1f2e")]},
               {"selector": "th", "props": [("background", "#5b4fcf"), ("color", "#ffffff"),
                    ("font-weight", "600"), ("padding", "6px 11px"), ("text-align", "left")]},
               {"selector": "td", "props": [("padding", "5px 11px"),
                    ("border-bottom", "1px solid #ececf3")]},
               {"selector": "", "props": [("border-collapse", "collapse"),
                    ("font-family", "Malgun Gothic, sans-serif"), ("font-size", "12.5px")]}]))
    return s.format(포맷 or {}, precision=1, thousands=",")

def 억(s): return s / 1e8      # 원 → 억원

## 1. 데이터 결합

세 자료를 시군구 × 업종 한 테이블로 합친다.

| 역할 | 자료 | 비고 |
|---|---|---|
| 분자 | BC 소비데이터 (결제금액 · 건수) | 대회 제공 · 2026.01~06 |
| 분모 ① | 업종별 점포 수 | 소상공인시장진흥공단 상가(상권)정보 |
| 분모 ② | 20대 이상 성인인구 | 행정안전부 주민등록인구 6개월 평균 |

**법인(GENDER_CD 4)만 제외**하고 내국인·외국인은 모두 포함한다 — 분모인 점포 수가
내·외국인 구분이 없는 전체 점포라 분자도 범위를 맞춰야 한다.

In [ ]:
bc = pd.read_csv("data/ABP_CONTEST_DATA.csv", encoding="utf-8",
                 dtype={"STRD_YYMM": str, "GENDER_CD": str, "AGE_CD": str, "TP_BUZ_NO": str})
pop = pd.read_csv("data/인구_시군구_성별_연령_202601_202606.csv", encoding="utf-8-sig",
                  dtype={"STRD_YYMM": str, "GENDER_CD": str, "AGE_CD": str, "행정구역코드": str})
업소 = pd.read_csv("data/cache/업소수_전국시군구.csv", encoding="utf-8-sig",
                 dtype={"TP_BUZ_NO": str})

# BC는 시도·시군구가 두 칸이라 인구 자료처럼 한 칸으로 합친다
bc["행정구역명"] = (bc["SIDO_NM"].str.strip() + " " + bc["CCG_NM"].str.strip()
                ).str.replace(r"\s+", " ", regex=True)
bc["행정구역명"] = bc["행정구역명"].replace("세종특별자치시 세종특별자치시", "세종특별자치시")

# 인천 행정구역 개편 — 점포 자료가 개편 후 체계라 중구·동구를 합친 단위만 대응된다
인천병합 = {"인천광역시 중구": "인천광역시 중구+동구", "인천광역시 동구": "인천광역시 중구+동구"}
bc["행정구역명"] = bc["행정구역명"].replace(인천병합)
pop["행정구역명"] = pop["행정구역명"].replace(인천병합)

분석업종 = sorted(업소["TP_BUZ_NO"].unique())   # 8개 — 대형할인점·갈비전문점·한정식은 점포 자료 미수록
개인 = ["1", "2", "3"]                          # 법인(4) 제외
성인 = ["2", "3", "4", "5", "6"]                # 20대 이상

소비 = (bc[bc["GENDER_CD"].isin(개인) & bc["TP_BUZ_NO"].isin(분석업종)]
        .groupby(["행정구역명", "TP_BUZ_NO", "TP_BUZ_NM"], as_index=False)
        .agg(amt=("amt", "sum"), cnt=("cnt", "sum")))

인구 = (pop[pop["AGE_CD"].isin(성인)]
        .groupby(["행정구역명", "STRD_YYMM"])["인구"].sum()
        .groupby("행정구역명").mean().rename("성인인구").reset_index())

분석 = (소비.merge(업소, on=["행정구역명", "TP_BUZ_NO"], how="inner")
            .merge(인구, on="행정구역명", how="inner"))
분석 = 분석[(분석["업소수"] > 0) & (분석["amt"] > 0)].reset_index(drop=True)

display(표(pd.DataFrame([
    ["시군구", f"{분석['행정구역명'].nunique()}개", "광주·전남 제외(상가정보 누락, 인구의 6.2%) · 인천 중구+동구 병합"],
    ["업종", f"{분석['TP_BUZ_NO'].nunique()}개", " · ".join(분석.drop_duplicates('TP_BUZ_NO')['TP_BUZ_NM'].str.replace(' ', ''))],
    ["분석 조합", f"{len(분석):,}개", "시군구 × 업종 · 점포 수와 결제액이 모두 0보다 큰 조합"],
], columns=["구분", "규모", "설명"]), "분석 범위"))

## 2. 업종별 회귀 — 침투지수

$$\log(	ext{BC 소비}) = eta_0 + eta_1 \log(	ext{점포 수}) + eta_2 \log(	ext{성인인구}) + arepsilon$$

**로그를 씌우는 이유** — 세 변수 모두 지역 간 최대 19만 배 차이 나는 극단적 우편향 분포다.
그대로 쓰면 대도시 몇 곳이 회귀선을 끌고 간다(상위 10% 지역의 오차 배율 2.3~4.2배 → 0.7~1.8배).
또 `소비 = A × 점포수^b × 인구^c` 라는 곱셈 구조를 선형으로 풀어 주고, 계수가 곧 탄력성이 된다.
무엇보다 **잔차가 그대로 침투지수**가 된다 — `exp(잔차) × 100`.

**업종마다 따로 적합**한다(8개). 하나로 묶으면 R²가 0.891에서 0.768로 떨어진다 —
업종마다 점포 규모와 객단가가 다르기 때문이다.

표준오차는 **HC3 로버스트**를 쓴다. 작은 지역일수록 예측이 더 흔들리는 이분산 때문이다.

| 지표 | 산식 |
|---|---|
| 모델예측 | 회귀 적합값 `exp(β₀ + β₁·log 점포수 + β₂·log 인구)` |
| **침투지수** | 실제 소비 ÷ 모델예측 × 100 |
| 격차금액 | 모델예측 − 실제 소비 |
| 시장 백분위 | 모델예측의 업종 내 백분위 |

In [ ]:
for c, 원 in [("log_소비", "amt"), ("log_업소수", "업소수"), ("log_인구", "성인인구")]:
    분석[c] = np.log(분석[원])
설명변수 = ["log_업소수", "log_인구"]

모음, 요약 = [], []
for 코드, d in 분석.groupby("TP_BUZ_NO"):
    모델 = sm.OLS(d["log_소비"], sm.add_constant(d[설명변수])).fit(cov_type="HC3")
    모음.append(d.assign(모델예측=np.exp(모델.fittedvalues)))
    요약.append({"업종": d["TP_BUZ_NM"].iat[0].replace(" ", ""), "지역수": len(d),
                 "R²": 모델.rsquared,
                 "점포수 계수": 모델.params["log_업소수"], "인구 계수": 모델.params["log_인구"],
                 "규모 탄력성 합": 모델.params["log_업소수"] + 모델.params["log_인구"]})

결과 = pd.concat(모음, ignore_index=True)
결과["침투지수"] = 결과["amt"] / 결과["모델예측"] * 100
결과["격차금액"] = 결과["모델예측"] - 결과["amt"]
결과["시장백분위"] = 결과.groupby("TP_BUZ_NO")["모델예측"].rank(pct=True) * 100

설명력 = pd.DataFrame(요약).sort_values("R²", ascending=False)
가중 = np.average(설명력["R²"], weights=설명력["지역수"])
display(표(설명력, f"업종별 회귀 — 가중평균 R² {가중:.3f}",
           {"R²": "{:.3f}", "점포수 계수": "{:.3f}", "인구 계수": "{:.3f}", "규모 탄력성 합": "{:.3f}"}))

**개별 계수는 해석하지 않는다.** `log(점포수)`와 `log(인구)`의 상관이 0.89~0.97, VIF가 4.8~16.5로
다중공선성이 있어 계수를 둘로 쪼개 읽으면 불안정하다. 다만 공선성은 **계수의 표준오차만
부풀리고 적합값·잔차에는 영향이 없어** 침투지수는 이 영향 밖에 있다
(부트스트랩 500회에서 개별 계수 변동 ±11~19% 대 침투지수 변동 ±3~6%).
해석에는 **규모 탄력성의 합**만 쓴다.

## 3. σ — 모델 오차 대비 얼마나 벗어났나

침투지수 70은 업종에 따라 의미가 다르다. 편의점처럼 모델이 잘 맞는 업종(예측오차 0.247)에서는
큰 이탈이지만, 일식회집(0.682)에서는 흔한 수준이다. 그래서 **업종별 예측오차로 나눠 표준화**한다.

$$\sigma = rac{\log(	ext{침투지수} \div 100)}{	ext{업종별 5-fold CV 예측오차}}$$

**폴드를 난수가 아니라 순차 분할(`idx[i::5]`)로 고정한다.** 난수로 나누면 업종별 CV 오차가
1%가량 흔들려 σ가 −1.5 경계에 있는 조합의 채택 여부가 실행할 때마다 바뀐다.

> σ는 **가설검정 통계량이 아니라 순위 지표**다. 순열검정은 잔차 크기 분포가 보존돼 귀무가설이
> 성립하지 않고, FDR은 업종당 227개라 경험적 p 하한(0.0044)이 BH 임계(0.00003)를 넘을 수 없다.
> 검증은 유의성이 아니라 **재현성**으로 한다 (7절).

In [ ]:
def CV예측오차(d, 설명변수, 폴드=5):
    """순차 분할 교차검증의 out-of-fold 잔차 표준편차. 실행할 때마다 같은 값이 나온다."""
    d = d.reset_index(drop=True)
    X = sm.add_constant(d[설명변수]).values
    y = d["log_소비"].values
    oof = np.full(len(d), np.nan)
    for i in range(폴드):
        검증 = np.arange(len(d))[i::폴드]
        학습 = np.setdiff1d(np.arange(len(d)), 검증)
        oof[검증] = X[검증] @ np.linalg.lstsq(X[학습], y[학습], rcond=None)[0]
    return np.std(y - oof, ddof=1), oof

예측오차, OOF = {}, {}
for 코드, d in 결과.groupby("TP_BUZ_NO"):
    예측오차[코드], OOF[코드] = CV예측오차(d, 설명변수)

결과["시그마"] = np.log(결과["침투지수"] / 100) / 결과["TP_BUZ_NO"].map(예측오차)

오차표 = (결과.drop_duplicates("TP_BUZ_NO")[["TP_BUZ_NO", "TP_BUZ_NM"]]
          .assign(예측오차=lambda d: d["TP_BUZ_NO"].map(예측오차))
          .assign(정상범위_하한=lambda d: np.exp(-d["예측오차"]) * 100,
                  정상범위_상한=lambda d: np.exp(d["예측오차"]) * 100)
          .sort_values("예측오차").drop(columns="TP_BUZ_NO")
          .rename(columns={"TP_BUZ_NM": "업종", "정상범위_하한": "정상범위 하한",
                           "정상범위_상한": "정상범위 상한"}))
오차표["업종"] = 오차표["업종"].str.replace(" ", "")
display(표(오차표, "업종별 예측오차와 정상범위 (±1σ)",
           {"예측오차": "{:.3f}", "정상범위 하한": "{:.1f}", "정상범위 상한": "{:.1f}"}))
print("정상범위 = 모델이 흔히 벗어나는 폭. 이 안의 침투지수는 저침투라 단정하기 어렵다.")

## 4. 후보 선정 — 네 조건을 모두 통과

점수를 매겨 순위를 낸 것이 아니라 **네 조건을 모두 통과해야 남는 방식**이다.
가중합을 쓰지 않았으므로 "왜 저 항목에 가중치를 더 줬냐"는 논쟁이 생기지 않는다.

| 단계 | 기준 | 왜 |
|---|---|---|
| ① 격차 | 침투지수 < 100 | 예상에 못 미치는 시장 |
| ② 시장 | 업종 내 시장 백분위 > 50 | 작은 시장은 고쳐도 남는 게 없다 |
| ③ 신뢰 | σ ≤ −1.5 | 모델 예측오차의 1.5배 넘게 낮다 |
| ④ 규모 | 격차금액 ≥ 20억 | 실행 가치 |

In [ ]:
격차경계, 시장경계, 신뢰경계, 규모하한 = 100, 50, -1.5, 20e8

m격차 = 결과["침투지수"] < 격차경계
m시장 = 결과["시장백분위"] > 시장경계
m신뢰 = 결과["시그마"] <= 신뢰경계
m규모 = 결과["격차금액"] >= 규모하한
누적 = [m격차, m격차 & m시장, m격차 & m시장 & m신뢰, m격차 & m시장 & m신뢰 & m규모]

깔때기 = pd.DataFrame(
    [["전체 조합", len(결과), 100.0]] +
    [[nm, int(m.sum()), m.sum() / len(결과) * 100] for nm, m in zip(
        ["① 침투지수 < 100", "② 시장 백분위 > 50", "③ σ ≤ −1.5", "④ 격차금액 ≥ 20억"], 누적)],
    columns=["단계", "남은 조합", "전체 대비 %"])
display(표(깔때기, "4단 필터", {"남은 조합": "{:,.0f}", "전체 대비 %": "{:.1f}%"}))

후보 = 결과[누적[-1]].sort_values("격차금액", ascending=False).reset_index(drop=True)
후보["신뢰등급"] = np.where(후보["시그마"] <= -2, "강", "중")

# 같은 지역 8개 업종 중 몇 개가 100 미만인가 — 지역 전체 약세인지 업종 한정인지 보는 참고값
지역약세 = 결과.groupby("행정구역명")["침투지수"].agg(지역100미만=lambda s: int((s < 100).sum()))
후보 = 후보.merge(지역약세, on="행정구역명")

# 6개월 모두 거래가 있었는지 — 극소 거래 조합이 끼지 않았는지 확인
거래월 = (bc[bc["GENDER_CD"].isin(개인) & (bc["amt"] > 0)]
          .groupby(["행정구역명", "TP_BUZ_NO"])["STRD_YYMM"].nunique().rename("거래월수"))
후보 = 후보.merge(거래월, on=["행정구역명", "TP_BUZ_NO"], how="left")

print(f"후보 {len(후보)}개 조합 · {후보['행정구역명'].nunique()}개 시군구 · "
      f"격차금액 합 {억(후보['격차금액'].sum()):,.0f}억")
print(f"거래가 6개월 미만인 후보: {int((후보['거래월수'] < 6).sum())}개")

display(표(후보.assign(**{"실제(억)": lambda d: 억(d["amt"]), "예측(억)": lambda d: 억(d["모델예측"]),
                        "격차(억)": lambda d: 억(d["격차금액"]),
                        "업종": lambda d: d["TP_BUZ_NM"].str.replace(" ", "")})
          [["행정구역명", "업종", "업소수", "실제(억)", "예측(억)", "격차(억)",
            "침투지수", "시장백분위", "시그마", "신뢰등급", "지역100미만"]],
          f"후보 {len(후보)}개 — 격차금액 순",
          {"시그마": "{:.2f}", "업소수": "{:,.0f}", "시장백분위": "{:.0f}"}))

### 경계 사례를 숨기지 않는다

**대전 서구 슈퍼마켓**은 침투지수 50.7 · 격차 111억으로 그 지역에서 가장 낮은데도 후보가 아니다.
네 기준 중 셋을 통과하고 **σ −1.4622로 −1.5에 0.038 미달**했다. 다만 대전 서구는 지역 단위
캠페인 대상이라 실행에는 포함된다.

## 5. 왜 약한가 — 건수와 건당금액으로 쪼갠다

결제액이 낮은 이유는 둘 중 하나다. **드물게 쓰거나**, **한 번에 적게 쓰거나**.

| 지표 | 산식 |
|---|---|
| 건수지수 | `log(건수) ~ log(점포수) + log(인구)` 예측 대비 실제 × 100 |
| 건당지수 | 지역 건당금액 ÷ 전국 동업종 건당금액 × 100 |

건수지수는 침투지수와 **같은 잣대**(회귀 예측 대비)로 맞췄다. 건당금액은 회귀를 쓸 이유가 없어
전국 동업종 평균과 직접 비교한다.

In [ ]:
분석["log_건수"] = np.log(분석["cnt"])
건수예측 = {}
for 코드, d in 분석.groupby("TP_BUZ_NO"):
    m = sm.OLS(d["log_건수"], sm.add_constant(d[설명변수])).fit(cov_type="HC3")
    건수예측.update(zip(zip(d["행정구역명"], d["TP_BUZ_NO"]), np.exp(m.fittedvalues)))

전국건당 = 분석.groupby("TP_BUZ_NO").apply(
    lambda d: d["amt"].sum() / d["cnt"].sum(), include_groups=False)

분해 = 후보[["행정구역명", "TP_BUZ_NM", "TP_BUZ_NO", "침투지수", "격차금액", "cnt", "amt"]].copy()
분해["건수지수"] = [c / 건수예측[(z, k)] * 100
                for z, k, c in zip(분해["행정구역명"], 분해["TP_BUZ_NO"], 분해["cnt"])]
분해["건당지수"] = (분해["amt"] / 분해["cnt"]) / 분해["TP_BUZ_NO"].map(전국건당) * 100

display(표(분해.assign(업종=lambda d: d["TP_BUZ_NM"].str.replace(" ", ""),
                     **{"격차(억)": lambda d: 억(d["격차금액"])})
          [["행정구역명", "업종", "격차(억)", "침투지수", "건수지수", "건당지수"]],
          "침투 부족의 구성 — 건수인가 건당금액인가"))

print(f"건수지수 {분해['건수지수'].min():.0f}~{분해['건수지수'].max():.0f}  "
      f"(전국 수준의 {분해['건수지수'].min()/100:.0%}~{분해['건수지수'].max()/100:.0%})")
print(f"건당지수 {분해['건당지수'].min():.0f}~{분해['건당지수'].max():.0f}  — 100 안팎, 전국과 비슷하다")

**적은 것은 금액이 아니라 횟수다.** 한 번 결제할 때 쓰는 금액은 전국과 비슷한데, 결제 횟수가
전국 수준의 3~7할이다. BC카드가 그 지역에서 **"가끔 쓰는 카드"**가 되어 있다는 뜻이다.

> 그래서 혜택 설계가 달라진다. "5만 원 이상 결제 시 할인"(금액형)이 아니라
> **"이번 달에 3번 이상 결제하면 혜택"(횟수형)**이 맞다.

## 6. 기회 유형 3가지 — 그 돈은 지금 어디 있나

침투지수 하나만 보면 **"BC를 안 쓰는 것"**과 **"그 소비가 다른 데로 간 것"**이 구분되지 않는다.
둘은 회수 가능성이 완전히 다르다. 그래서 같은 지역을 **3층으로 넓혀 가며** 본다.

| 층 | 지수 | 묻는 것 |
|---|---|---|
| ① 업종 | 침투지수 | 이 업종이 약한가 |
| ② 카테고리 | 장보기(슈퍼·편의점·대형할인점) 또는 외식 8개 묶음 | 옆 업종으로 갔나 |
| ③ 지역 전체 | 그 지역 전체 1인당 BC 결제액 | 지역 자체가 약한가 |

②③은 모두 **1인당 BC 결제액 지수(전국=100)**다.

```
지역지수 < 80                    → 유형① 지역 전체 기회   (3층 모두 낮다)
카테고리÷지역 ≥ 80               → 유형② 업종 간 이동     (카테고리는 지역만큼 나온다)
그 외                            → 유형③ 채널 이동       (카테고리만 유독 꺼졌다)
```

In [ ]:
장보기 = ["4020", "4004", "4010"]                                       # 슈퍼마켓·대형할인점·편의점
외식 = ["8001", "8002", "8003", "8004", "8005", "8006", "8021", "8301"]

b = bc[bc["GENDER_CD"].isin(개인) & bc["행정구역명"].isin(결과["행정구역명"].unique())]
성인인구 = 결과.groupby("행정구역명")["성인인구"].first()

def 한명당지수(코드들=None):
    """지역별 1인당 BC 결제액을 전국 평균=100으로 지수화"""
    sub = b if 코드들 is None else b[b["TP_BUZ_NO"].isin(코드들)]
    s = sub.groupby("행정구역명")["amt"].sum().reindex(성인인구.index).fillna(0)
    return (s / 성인인구) / (s.sum() / 성인인구.sum()) * 100

지역지수, 장보기지수, 외식지수, 대형지수 = (한명당지수(), 한명당지수(장보기),
                                     한명당지수(외식), 한명당지수(["4004"]))

진단 = 후보[["행정구역명", "TP_BUZ_NM", "TP_BUZ_NO", "침투지수", "격차금액", "시그마"]].copy()
진단["지역지수"] = 지역지수.reindex(진단["행정구역명"]).values
진단["카테고리지수"] = [장보기지수[z] if k in 장보기 else 외식지수[z]
                   for z, k in zip(진단["행정구역명"], 진단["TP_BUZ_NO"])]
진단["카테고리÷지역"] = 진단["카테고리지수"] / 진단["지역지수"] * 100
진단["대형할인점"] = 대형지수.reindex(진단["행정구역명"]).values

유형명 = {1: "① 지역 전체 기회", 2: "② 업종 간 이동", 3: "③ 채널 이동"}

def 유형판정(r):
    if r["지역지수"] < 80:          return 1   # 지역 자체가 약하다 — 3층 모두 낮음
    if r["카테고리÷지역"] >= 80:    return 2   # 카테고리는 지역 수준만큼 나온다 — 옆 업종으로 갔다
    return 3                                  # 카테고리만 유독 꺼졌다 — 채널 밖으로 나갔을 가능성

진단["유형"] = 진단.apply(유형판정, axis=1)
진단["유형명"] = 진단["유형"].map(유형명)

display(표(진단.sort_values(["유형", "격차금액"], ascending=[True, False])
          .assign(업종=lambda d: d["TP_BUZ_NM"].str.replace(" ", ""),
                  **{"격차(억)": lambda d: 억(d["격차금액"])})
          [["유형명", "행정구역명", "업종", "격차(억)", "침투지수",
            "지역지수", "카테고리지수", "카테고리÷지역", "대형할인점"]],
          "3층 진단과 유형 판정", {"시그마": "{:.2f}"}))

In [ ]:
유형요약 = (진단.groupby(["유형", "유형명"])
            .agg(조합수=("격차금액", "size"), 격차합=("격차금액", "sum"),
                 지역_최소=("지역지수", "min"), 지역_최대=("지역지수", "max"),
                 비율_최소=("카테고리÷지역", "min"), 비율_최대=("카테고리÷지역", "max"))
            .reset_index())
유형요약["격차(억)"] = 억(유형요약["격차합"])
유형요약["지역지수"] = [f"{a:.0f}~{b_:.0f}" for a, b_ in
                   zip(유형요약["지역_최소"], 유형요약["지역_최대"])]
유형요약["카테고리÷지역"] = [f"{a:.0f}~{b_:.0f}" for a, b_ in
                      zip(유형요약["비율_최소"], 유형요약["비율_최대"])]
유형요약["회수 가능성"] = ["가장 큼 — 돈이 BC 밖에 있을 가능성",
                     "중간 — 이미 BC 안에서 다른 업종으로 결제 중일 수 있다",
                     "낮음 — 회수 가능 여부 자체가 미검증"]
display(표(유형요약[["유형명", "조합수", "격차(억)", "지역지수", "카테고리÷지역", "회수 가능성"]],
          f"기회 유형 3가지 — 격차 {억(진단['격차금액'].sum()):,.0f}억은 한 덩어리가 아니다",
          {"격차(억)": "{:,.0f}"}))

**유형②가 함정이다.** 동네 슈퍼 결제가 늘어도 대형마트 결제가 그만큼 줄면 BC카드 전체 매출은
그대로다. 그래서 이런 곳은 업종 하나만 보지 않고 **장보기 카테고리 합계**로 판정해야 한다.

### 시범 3곳

유형별 대표성 · 격차 규모 · 진단 가능성으로 골랐다. 세 곳 모두 **자기 유형 안에서 격차가 가장 큰**
조합이기도 하다.

| 시범 | 유형 | 왜 이 곳이 대표인가 |
|---|---|---|
| **대전 서구** | ① 지역 전체 기회 | 8개 업종이 전부 100 미만 — 지역 단위 진단이 가장 선명하다 |
| **진주** | ② 업종 간 이동 | 돈이 간 곳(대형할인점)이 특정되는 유일한 곳 |
| **서초** | ③ 채널 이동 | 유형 ③ 단독 · σ −2.73으로 12개 중 이탈이 가장 크다 |

## 7. 검증

σ가 가설검정 통계량이 아니므로 유의성 검정 대신 **"다른 조건에서도 같은 결론이 나오는가"**로
검증한다.

### 7-1. 홀드아웃 3종

후보 12개가 **자기 자신이 만든 모델 때문에** 뽑힌 것은 아닌지 확인한다.

| 방식 | 빼는 것 | 묻는 것 |
|---|---|---|
| 5-fold CV | 자기 자신 | 자기를 빼고 적합해도 여전히 낮은가 |
| 시도 홀드아웃 | 자기가 속한 시도 전체 | 강원 전체를 빼고 만든 모델로도 춘천이 낮은가 |
| 시간 분할 | 뒤 3개월 | 1~3월로 만든 모델이 4~6월도 맞히는가 |

In [ ]:
후보키 = set(zip(후보["행정구역명"], 후보["TP_BUZ_NO"]))

def 유지수(침투맵, 이름):
    """후보 12개 중 홀드아웃 예측에서도 침투지수 100 미만인 조합 수"""
    v = np.array([침투맵[k] for k in 후보키])
    return {"방식": 이름, "후보 유지": f"{int((v < 100).sum())} / {len(후보키)}",
            "평균 침투지수": v.mean(), "최대": v.max()}

행 = []
# (1) 5-fold CV — 자기 자신을 빼고 예측
침투 = {}
for 코드, d in 결과.groupby("TP_BUZ_NO"):
    d = d.reset_index(drop=True)
    침투.update({(z, 코드): a / np.exp(p) * 100
                for z, a, p in zip(d["행정구역명"], d["amt"], OOF[코드])})
행.append(유지수(침투, "5-fold 교차검증"))

# (2) 시도 홀드아웃 — 자기가 속한 시도 전체를 빼고 적합
결과["시도"] = 결과["행정구역명"].str.split().str[0]
침투 = {}
for 코드, d in 결과.groupby("TP_BUZ_NO"):
    for 시도, te in d.groupby("시도"):
        tr = d[d["시도"] != 시도]
        if len(tr) < 20: continue
        m = sm.OLS(tr["log_소비"], sm.add_constant(tr[설명변수])).fit()
        p = m.predict(sm.add_constant(te[설명변수], has_constant="add"))
        침투.update({(z, 코드): a / np.exp(v) * 100
                    for z, a, v in zip(te["행정구역명"], te["amt"], p)})
행.append(유지수(침투, "시도 단위 홀드아웃"))

# (3) 시간 분할 — 1~3월로 적합해 4~6월을 예측
반기 = {}
for 라벨, 월들 in [("전반", ["202601", "202602", "202603"]), ("후반", ["202604", "202605", "202606"])]:
    반기[라벨] = (bc[bc["GENDER_CD"].isin(개인) & bc["STRD_YYMM"].isin(월들)
                   & bc["TP_BUZ_NO"].isin(분석업종)]
                 .groupby(["행정구역명", "TP_BUZ_NO"])["amt"].sum())
침투 = {}
for 코드, d in 결과.groupby("TP_BUZ_NO"):
    idx = list(zip(d["행정구역명"], [코드] * len(d)))
    전, 후 = 반기["전반"].reindex(idx).values, 반기["후반"].reindex(idx).values
    ok = (전 > 0) & (후 > 0)
    m = sm.OLS(np.log(전[ok]), sm.add_constant(d[설명변수].values[ok])).fit()
    p = m.predict(sm.add_constant(d[설명변수].values[ok], has_constant="add"))
    침투.update({(z, 코드): a / np.exp(v) * 100
                for z, a, v in zip(d["행정구역명"].values[ok], 후[ok], p)})
행.append(유지수(침투, "시간 분할 (1~3월 → 4~6월)"))

display(표(pd.DataFrame(행), "홀드아웃 3종 — 후보 12개가 모두 유지되는가",
           {"평균 침투지수": "{:.1f}", "최대": "{:.1f}"}))

**세 경우 모두 12개가 그대로 남는다.** 시도 홀드아웃에서는 오히려 침투지수가 더 낮게 나와
현재 수치가 **보수적인 쪽**임을 확인했다.

### 7-2. 컷오프 민감도

σ와 격차 기준을 흔들어 본다. 기준을 우리에게 유리하게 잡아 후보를 만들어 낸 것은 아닌지 본다.

In [ ]:
민감도 = pd.DataFrame(
    [[f"σ ≤ {s}"] + [int(((결과["침투지수"] < 100) & (결과["시장백분위"] > 50) &
                         (결과["시그마"] <= s) & (결과["격차금액"] >= g * 1e8)).sum())
                    for g in [10, 20, 30, 50]]
     for s in [-1.3, -1.4, -1.5, -1.6, -1.8, -2.0]],
    columns=["신뢰 기준", "격차 10억", "격차 20억", "격차 30억", "격차 50억"])
display(표(민감도, "컷오프 민감도 — 기준을 흔들면 후보가 몇 개가 되나 (현행: σ ≤ −1.5 · 20억)"))

**기준을 풀면 12개가 전부 유지된다** — 느슨하게 잡아서 나온 후보가 아니다.
다만 **조이면 줄어든다**(σ −1.6에서 8개, −2.0에서 3개). 따라서 *"어떤 기준에서도 12개가 남는다"*는
표현은 쓸 수 없고, **"12개가 전부가 아니라 12개가 1순위"**라고 서술한다.
σ −1.3까지 풀면 13곳이 추가된다.

### 7-3. 재현성

반기를 나눠 각각 다시 산출한 σ의 상관이 **0.946**이다. 시점을 바꿔도 같은 곳이 같은 순위로 나온다.

In [ ]:
행 = []
for 라벨 in ["전반", "후반"]:
    모음 = []
    for 코드, d in 결과.groupby("TP_BUZ_NO"):
        idx = list(zip(d["행정구역명"], [코드] * len(d)))
        a = 반기[라벨].reindex(idx).values
        ok = a > 0
        d2 = d[ok].assign(log_소비=np.log(a[ok]))
        m = sm.OLS(d2["log_소비"], sm.add_constant(d2[설명변수])).fit()
        오차, _ = CV예측오차(d2, 설명변수)
        모음.append(pd.DataFrame({
            "키": list(zip(d2["행정구역명"], d2["TP_BUZ_NO"])),
            라벨: (d2["log_소비"].values - m.fittedvalues.values) / 오차}))
    행.append(pd.concat(모음, ignore_index=True).set_index("키"))

반기σ = 행[0].join(행[1], how="inner")
print(f"반기 재산출 σ 상관: {반기σ['전반'].corr(반기σ['후반']):.3f}   (n={len(반기σ):,})")
print(f"후보 12개만: {반기σ.loc[[k for k in 후보키 if k in 반기σ.index]].corr().iloc[0, 1]:.3f}")

## 한계

- **다른 카드사 실적을 모른다.** "BC만 약한 것"인지 "그 지역이 카드 자체를 덜 쓰는 것"인지
  구분할 수 없다. 여신금융협회 집계는 전국 단위라 지역별 비교에 쓸 수 없다
- **온라인 결제가 따로 구분되지 않는다.** 유형③의 "채널 밖으로 나갔다"는 추정이다
- **광주·전남이 빠졌다.** 상가정보에 점포 자료가 없다 (인구의 6.2%)
- **왜 안 쓰는지는 모른다.** 이 분석은 "어디가 약한가"까지만 답한다 — 시범 3곳을 운영하는 이유다